In [267]:
# | default_exp dmd_scaling_dataset

# Scaling the MLP

> We scale a neural network by freezing it and adding new trainable layers.

#### Imports and Setup

In [268]:
from ast import literal_eval
from copy import deepcopy

import graphviz
import matrepr
import torch
import torchinfo
from safetensors.torch import load_model, save_model
from torch import nn
from torch.nn import (
    CrossEntropyLoss,
    KLDivLoss,
    L1Loss,
    MSELoss,
    CosineSimilarity,
    HuberLoss,
)
from torch.nn import functional as F
from torch.optim import SGD, Adam, AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from learningdynamics.models import MLP, init_weights
from learningdynamics.utils import (
    compute_model_accuracy,
    getActivation,
    safetensors_metadata_parser,
)

In [269]:
# | hide
matrepr.params.title = True
matrepr.params.indices = False
matrepr.params.precision = 4
matrepr.cell_align = "left"
graphviz.set_jupyter_format("png")

%matplotlib inline

#### Load model trained on binary classification.

In [270]:
file_path = "../saved_weights/mnist_model.safetensors"

# Parse metadata
metadata = safetensors_metadata_parser(file_path=file_path)

# Load model
model = MLP(
    config=literal_eval(metadata["config"]),
    nonlinearity=metadata["nonlinearity"],
    in_features=784,
    out_features=10,
)
_ = load_model(model, file_path, device="cuda:0")

#### Setup the dataset

In [271]:
transform = transforms.Compose(
    [
        transforms.ToTensor(),  # first, convert image to PyTorch tensor
        transforms.Normalize((0.1307,), (0.3081,)),  # normalize inputs
    ]
)
train_set = datasets.MNIST(
    root="/mnt/datasets/", train=True, download=True, transform=transform
)
test_set = datasets.MNIST(
    root="/mnt/datasets/", train=False, download=True, transform=transform
)

#### Show model and test accuracy

In [272]:
print(f"Testing Accuracy: {compute_model_accuracy(model, test_set)}")

Testing Accuracy: 0.9710999727249146


In [273]:
# torchinfo.summary(model, row_settings=["var_names", "ascii_only"])
model

MLP(
  (features): Sequential(
    (0): Linear(in_features=784, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=16, bias=True)
    (5): ReLU()
    (6): Linear(in_features=16, out_features=10, bias=True)
  )
)

#### Scale Model

In [274]:
def freeze_and_scale_model(
    model,
    start,
    layers_to_add=1,
    device="cuda:0",
):
    # Set up scaled model
    model.to(device)
    scaled_model = deepcopy(model)

    # Update model
    scaled_container = torch.nn.Sequential()
    for idx, module in enumerate(scaled_model.features):
        # Copy over weights and biases
        if hasattr(module, "weight"):
            module.weight.requires_grad_(False)
        if hasattr(module, "bias"):
            module.bias.requires_grad_(False)
        scaled_container.append(module)
        # Add new layers
        if idx == start:
            prev_out = model.features[idx - 1].out_features
            for _ in range(layers_to_add):
                linear_layer = nn.Linear(prev_out, prev_out)
                linear_layer.apply(init_weights)
                scaled_container.append(linear_layer)
                scaled_container.append(nn.ReLU())
    scaled_model.features = scaled_container
    scaled_model.to(device=device)

    scaled_model.to(device="cpu")
    return scaled_model

#### Control panel

In [275]:
DEVICE = "cuda:0"
PRINT_FREQ = 1

NUM_LAYERS_ADDED = 10
LAYER_START = 3
DIMENSIONALITY = model.features[LAYER_START + 1].in_features
LAYER_END = LAYER_START + 2

BATCH_SIZE = 4_096
NUM_EPOCHS = 200
LEARNING_RATE = 1e-2

dataloader = DataLoader(dataset=train_set, shuffle=True, batch_size=BATCH_SIZE)

In [276]:
scaled_model = freeze_and_scale_model(
    model,
    start=LAYER_START,
    layers_to_add=NUM_LAYERS_ADDED,
)

In [277]:
torchinfo.summary(scaled_model, row_settings=["var_names", "ascii_only"])
# scaled_model

Layer (type (var_name))                  Param #
MLP (MLP)                                --
+ Sequential (features)                  --
|    + Linear (0)                        (200,960)
|    + ReLU (1)                          --
|    + Linear (2)                        (8,224)
|    + ReLU (3)                          --
|    + Linear (4)                        1,056
|    + ReLU (5)                          --
|    + Linear (6)                        1,056
|    + ReLU (7)                          --
|    + Linear (8)                        1,056
|    + ReLU (9)                          --
|    + Linear (10)                       1,056
|    + ReLU (11)                         --
|    + Linear (12)                       1,056
|    + ReLU (13)                         --
|    + Linear (14)                       1,056
|    + ReLU (15)                         --
|    + Linear (16)                       1,056
|    + ReLU (17)                         --
|    + Linear (18)                    

In [278]:
print(f"Testing Accuracy: {compute_model_accuracy(scaled_model.to('cpu'), test_set)}")

Testing Accuracy: 0.09740000218153


#### Train scaled model

In [279]:
activation_dict = {}
hooks = []

In [280]:
# Setup training
reconstruct_criterion = MSELoss()
l1_criterion = L1Loss()
classification_criterion = CrossEntropyLoss()
kl_criterion = KLDivLoss(reduction="batchmean")
cos_criterion = CosineSimilarity()
huber_criterion = HuberLoss(delta=2)
optimizer = SGD(
    scaled_model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4, momentum=0.9
)
# optimizer = AdamW(scaled_model.parameters(), lr=LEARNING_RATE, weight_decay=0.001)
scheduler = ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3, threshold=0.001, verbose=True
)

In [281]:
scaled_model.to(DEVICE)
scaled_model.train()

# Attach hook to original
hooks.append(
    model.features[LAYER_END].register_forward_hook(
        getActivation(f"layer_{LAYER_END}", activation_dict)
    )
)

# Attach hook to scaled
new_end = LAYER_END + NUM_LAYERS_ADDED * 2
hooks.append(
    scaled_model.features[new_end].register_forward_hook(
        getActivation(f"scaled_layer_{new_end}", activation_dict, detach=False)
    )
)

# Run training
loss_vector = []
for epoch in range(NUM_EPOCHS):
    running_loss = 0.0  # Track loss within the epoch

    # Iterate through dataloader
    for input, label in dataloader:
        # Feed input in to get activations
        input, label = input.to(DEVICE), torch.flatten(label.to(DEVICE))
        output = scaled_model(input)

        with torch.no_grad():
            _ = model(input)

        # Zero out optimizer
        optimizer.zero_grad()

        # Grab activations
        output_activation = activation_dict[f"scaled_layer_{new_end}"].to(DEVICE)
        target_activation = activation_dict[f"layer_{LAYER_END}"].to(DEVICE)

        mse_loss = reconstruct_criterion(output_activation, target_activation)
        classification_loss = classification_criterion(output, label.long())
        kl_loss = kl_criterion(
            F.log_softmax(output_activation, dim=1),
            F.softmax(target_activation, dim=1),
        )
        l1_loss = l1_criterion(output_activation, target_activation)
        cos_loss = torch.mean(cos_criterion(output_activation, target_activation))
        huber_loss = huber_criterion(output_activation, target_activation)

        alpha, beta = 1, 0
        loss = alpha * classification_loss + beta * mse_loss
        # loss = mse_loss
        loss.backward()
        optimizer.step()

        running_loss += loss.item()  # Accumulate loss

    # print(output_activation)
    # print(target_activation)

    epoch_loss = running_loss / len(dataloader)  # Average loss over all batches
    loss_vector.append(epoch_loss)  # Store loss for later analysis

    # Step the scheduler
    scheduler.step(epoch_loss)

    epoch_loss = running_loss / len(dataloader)  # Average loss over all batches
    loss_vector.append(epoch_loss)  # Store loss for later analysis5

    if (epoch + 1) % PRINT_FREQ == 0:
        print(f"Epoch {epoch + 1}/{NUM_EPOCHS}, Loss: {epoch_loss:.4f}")

# Remove hook
for hook in hooks:
    hook.remove()

Epoch 1/200, Loss: 2.2922
Epoch 2/200, Loss: 2.1937
Epoch 3/200, Loss: 1.9860
Epoch 4/200, Loss: 1.6891
Epoch 5/200, Loss: 1.1177
Epoch 6/200, Loss: 0.7751
Epoch 7/200, Loss: 0.5169
Epoch 8/200, Loss: 0.3750
Epoch 9/200, Loss: 0.2833
Epoch 10/200, Loss: 0.2507
Epoch 11/200, Loss: 0.2189
Epoch 12/200, Loss: 0.1953
Epoch 13/200, Loss: 0.1881
Epoch 14/200, Loss: 0.1769
Epoch 15/200, Loss: 0.1669
Epoch 16/200, Loss: 0.1579
Epoch 17/200, Loss: 0.1522
Epoch 18/200, Loss: 0.1491
Epoch 19/200, Loss: 0.1534
Epoch 20/200, Loss: 0.1450
Epoch 21/200, Loss: 0.1392
Epoch 22/200, Loss: 0.1306
Epoch 23/200, Loss: 0.1300


KeyboardInterrupt: 

In [282]:
print(
    f"Testing Accuracy: {compute_model_accuracy(scaled_model.to('cpu').eval(), test_set)}"
)

Testing Accuracy: 0.9599000215530396


#### Update config

In [283]:
config = literal_eval(metadata["config"])
insert = [model.features[LAYER_START - 1].out_features] * NUM_LAYERS_ADDED
config[LAYER_START // 2 : LAYER_START // 2] = insert

In [284]:
metadata_dict = {
    "config": str(config),
    "dataset": metadata["dataset"],
    "nonlinearity": metadata["nonlinearity"],
    "added_layers": str(NUM_LAYERS_ADDED),
    "layer_start": str(LAYER_START),
    "layer_in_out": str(DIMENSIONALITY),
}

save_model(
    scaled_model,
    "../saved_weights/mnist_scaled_model.safetensors",
    metadata=metadata_dict,
)